In [ ]:
import pandas as pd, geopandas as gpd
from pyproj import Transformer
import os

- 노선 확인

In [ ]:
T = ['공주시', '고창군', '김천시']

In [ ]:
## 읍면동 경계 52개 (sub)
line = pd.read_excel('/content/센서스 공간정보 지역 코드 행정동.xlsx', header=1, dtype=str)
line['adm_cd8'] = line['시도코드'].str.zfill(2) + line['시군구코드'].str.zfill(3) + line['읍면동코드'].str.zfill(3)
want = line.loc[line['시군구명칭'].isin(T), 'adm_cd8']
gdf = gpd.read_file('/content/BND_ADM_DONG_PG.shp', encoding='cp949')
gdf['adm_cd8'] = gdf['ADM_CD'].astype(str).str[:8]
sub = gdf[gdf['adm_cd8'].str[:5].isin(want.str[:5].unique())].copy().to_crs('EPSG:5179')
sub = sub.merge(line[['adm_cd8', '시군구명칭', '읍면동명칭']], on='adm_cd8', how='left')

In [ ]:
## 병원 6곳 (hg)
h = pd.read_csv('/content/건강_병원_전국.csv', encoding='cp949')
h['주소'] = h['도로명주소'].fillna('') + ' ' + h['지번주소'].fillna('')
h['시군구'] = h['주소'].str.extract(r'(공주시|고창군|김천시)')[0]
tgt = h[h['업태구분명'].isin(['병원', '종합병원']) & (h['영업상태명'] == '영업/정상') & h['시군구'].notna()].copy()
tgt['gx'], tgt['gy'] = tgt['좌표정보(X)'].round(0), tgt['좌표정보(Y)'].round(0)
tgt = tgt.drop_duplicates(subset=['시군구', 'gx', 'gy']).reset_index(drop=True)
tr = Transformer.from_crs('EPSG:5174', 'EPSG:4326', always_xy=True)
tgt['lon'], tgt['lat'] = tr.transform(tgt['좌표정보(X)'].values, tgt['좌표정보(Y)'].values)
hg = gpd.GeoDataFrame(tgt, geometry=gpd.points_from_xy(tgt['lon'], tgt['lat']), crs='EPSG:4326').to_crs('EPSG:5179')
print('읍면동', len(sub), '(기대 52) / 병원', len(hg), '(기대 6)')

In [ ]:
## 노선 점검
rs = pd.read_csv('/content/routes_stops.csv', dtype={'route_id': str, 'route_no': str, 'stop_id': str})
g = gpd.GeoDataFrame(rs, geometry=gpd.points_from_xy(rs['lon'], rs['lat']), crs='EPSG:4326').to_crs('EPSG:5179')
u = g.drop_duplicates(['sigun', 'stop_id']).reset_index(drop=True)

In [ ]:
# 정류장이 자기 시군 읍면동 경계 안에 있는 비율
j = gpd.sjoin(u, sub[['adm_cd8', '시군구명칭', 'geometry']], how='left', predicate='within')
j = j.drop_duplicates(['sigun', 'stop_id'])
j['own'] = j['시군구명칭'] == j['sigun']
print('\n(c) 자기 시군 경계 안 정류장')
print(j.groupby('sigun')['own'].agg(안=('sum'), 전체=('count'), 비율=('mean')))

In [ ]:
# 병원 300m 안 정류장과 지나는 노선번호 수
rows = []
for _, r in hg.iterrows():
    cand = u[u['sigun'] == r['시군구']]
    ids = set(cand[cand.geometry.distance(r.geometry) <= 300]['stop_id'])
    n = rs[(rs['sigun'] == r['시군구']) & rs['stop_id'].isin(ids)]['route_no'].nunique()
    rows.append({'시군구': r['시군구'], '병원': r['사업장명'], '300m내 정류장': len(ids), '지나는 노선번호 수': n})
print('\n(d) 병원 접근 노선')
print(pd.DataFrame(rows).to_string())

# 노선 하나씩 제거했을 때 병원 연결을 잃는 읍면과 65세 이상 인구를 계산

In [ ]:
import re
from collections import defaultdict
import numpy as np
from scipy.spatial import cKDTree

In [ ]:
MERGE_M, HOSP_R = 60, 300
LABEL = {0: '직통', 1: '환승1회', 9: '불가'}

In [ ]:
#읍면동(52개) + 지표
line = pd.read_excel('/content/센서스 공간정보 지역 코드 행정동.xlsx', header=1, dtype=str)
line['adm_cd8'] = line['시도코드'].str.zfill(2) + line['시군구코드'].str.zfill(3) + line['읍면동코드'].str.zfill(3)
want = line.loc[line['시군구명칭'].isin(T), 'adm_cd8']
gdf = gpd.read_file('/content/BND_ADM_DONG_PG.shp', encoding='cp949')
gdf['adm_cd8'] = gdf['ADM_CD'].astype(str).str[:8]
sub = gdf[gdf['adm_cd8'].str[:5].isin(want.str[:5].unique())].copy().to_crs('EPSG:5179')
sub = sub.merge(line[['adm_cd8', '시군구명칭', '읍면동명칭']], on='adm_cd8', how='left')

pop = pd.read_csv('/content/행정안전부_지역별(행정동) 성별 연령별 주민등록 인구수_20260930.csv', encoding='cp949')
pop = pop[pop['시군구명'].isin(T)].copy()
cols65 = [c for c in pop.columns if re.match(r'^\d+세', c) and int(re.match(r'^(\d+)세', c).group(1)) >= 65]
for c in cols65 + ['계']:
    pop[c] = pd.to_numeric(pop[c].astype(str).str.replace(',', ''), errors='coerce')
pop['pop65'] = pop[cols65].sum(axis=1)
pop['ratio65'] = pop['pop65'] / pop['계']
pop = pop.rename(columns={'시군구명': '시군구명칭', '읍면동명': '읍면동명칭'})[['시군구명칭', '읍면동명칭', 'pop65', 'ratio65']]
sub = sub.merge(pop, on=['시군구명칭', '읍면동명칭'], how='left')

st = pd.read_csv('/content/국토교통부_전국 버스정류장 위치정보_20251031.csv', encoding='cp949')
st = st[st['도시명'].astype(str).str.contains('공주|고창|김천')]
stg = gpd.GeoDataFrame(st, geometry=gpd.points_from_xy(st['경도'], st['위도']), crs='EPSG:4326').to_crs('EPSG:5179')

h = pd.read_csv('/content/건강_병원_전국.csv', encoding='cp949')
h['주소'] = h['도로명주소'].fillna('') + ' ' + h['지번주소'].fillna('')
h['시군구'] = h['주소'].str.extract(r'(공주시|고창군|김천시)')[0]
tgt = h[h['업태구분명'].isin(['병원', '종합병원']) & (h['영업상태명'] == '영업/정상') & h['시군구'].notna()].copy()
tgt['gx'], tgt['gy'] = tgt['좌표정보(X)'].round(0), tgt['좌표정보(Y)'].round(0)
tgt = tgt.drop_duplicates(subset=['시군구', 'gx', 'gy']).reset_index(drop=True)
tr = Transformer.from_crs('EPSG:5174', 'EPSG:4326', always_xy=True)
tgt['lon'], tgt['lat'] = tr.transform(tgt['좌표정보(X)'].values, tgt['좌표정보(Y)'].values)
hg = gpd.GeoDataFrame(tgt, geometry=gpd.points_from_xy(tgt['lon'], tgt['lat']), crs='EPSG:4326').to_crs('EPSG:5179')

rp = sub[['adm_cd8', '시군구명칭', 'geometry']].copy()
rp['geometry'] = rp.geometry.representative_point()
near = gpd.sjoin_nearest(rp[['adm_cd8', 'geometry']], stg[['geometry']], distance_col='d_near_m')
sub = sub.merge(near.groupby('adm_cd8')['d_near_m'].min().round().rename('d_near_m'), on='adm_cd8', how='left')
hk = []
for _, r in rp.iterrows():
    cand = hg[hg['시군구'] == r['시군구명칭']]
    hk.append({'adm_cd8': r['adm_cd8'], 'hosp_km': cand.geometry.distance(r.geometry).min() / 1000})
sub = sub.merge(pd.DataFrame(hk), on='adm_cd8', how='left')
print('읍면동', len(sub), '/ 병원', len(hg), '/ 인구 결측', sub['pop65'].isna().sum())

In [ ]:
# 노선망
def norm_name(s):
    return re.sub(r'\s+', '', re.sub(r'\(.*?\)', '', str(s)))

df = pd.read_csv('/content/routes_stops.csv', dtype={'route_id': str, 'route_no': str, 'stop_id': str})
df['route_no'] = df['route_no'].fillna(df['route_id'])
df['line'] = df['route_no'].astype(str)
df = df.dropna(subset=['lat', 'lon', 'stop_id'])

def build_network(df, merge_radius=MERGE_M):
    g = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df['lon'], df['lat']), crs='EPSG:4326').to_crs('EPSG:5179')
    g['x'], g['y'] = g.geometry.x, g.geometry.y
    s = g.drop_duplicates(['sigun', 'stop_id'])[['sigun', 'stop_id', 'stop_name', 'x', 'y']].reset_index(drop=True)
    s['nname'] = s['stop_name'].map(norm_name)
    parent = list(range(len(s)))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    for _, grp in s.groupby('sigun'):
        pts, names, ids = grp[['x', 'y']].to_numpy(), grp['nname'].to_numpy(), grp.index.to_numpy()
        for i, j in cKDTree(pts).query_pairs(merge_radius):
            if names[i] == names[j]:
                ra, rb = find(ids[i]), find(ids[j])
                if ra != rb:
                    parent[rb] = ra
    s['node'] = [f'{sg}:{find(i)}' for i, sg in zip(s.index, s['sigun'])]
    node_xy = s.groupby('node').agg(sigun=('sigun', 'first'), x=('x', 'mean'), y=('y', 'mean'), name=('stop_name', 'first')).reset_index()
    link = g[['sigun', 'line', 'stop_id']].merge(s[['sigun', 'stop_id', 'node']], on=['sigun', 'stop_id'], how='left')
    link['rkey'] = link['sigun'] + '|' + link['line']
    link = link.drop_duplicates(['rkey', 'node'])
    return node_xy, link.groupby('rkey')['node'].agg(set).to_dict()

def assign_nodes(node_xy, eup, hosp, hosp_r=HOSP_R):
    ng = gpd.GeoDataFrame(node_xy, geometry=gpd.points_from_xy(node_xy['x'], node_xy['y']), crs='EPSG:5179')
    j = gpd.sjoin(ng, eup[['adm_cd8', '시군구명칭', 'geometry']], how='inner', predicate='within')
    j = j[j['sigun'] == j['시군구명칭']]
    region_nodes = {sg: {a: set(gg['node']) for a, gg in grp.groupby('adm_cd8')} for sg, grp in j.groupby('sigun')}
    hosp_nodes = {}
    for sg, hs in hosp.groupby('시군구'):
        cand = ng[ng['sigun'] == sg]
        sel = set()
        for geom in hs.geometry:
            sel |= set(cand.loc[cand.geometry.distance(geom) <= hosp_r, 'node'])
        hosp_nodes[sg] = sel
    return region_nodes, hosp_nodes

def level_for_regions(region_nodes, hosp_nodes, nodes_of):
    ro = defaultdict(set)
    for r, ns in nodes_of.items():
        for n in ns:
            ro[n].add(r)
    r_h = set()
    for n in hosp_nodes:
        r_h |= ro.get(n, set())
    adj = defaultdict(set)
    for rs in ro.values():
        for a in rs:
            adj[a] |= rs
    r_h1 = set()
    for r in r_h:
        r_h1 |= adj[r]
    out = {}
    for e, ns in region_nodes.items():
        r_e = set()
        for n in ns:
            r_e |= ro.get(n, set())
        direct = r_e & r_h
        out[e] = (0, direct) if direct else ((1, set()) if r_e & r_h1 else (9, set()))
    return out

def simulate_line_removal(region_nodes, hosp_nodes, nodes_of, pop65):
    base = level_for_regions(region_nodes, hosp_nodes, nodes_of)
    rows = []
    for r in sorted(nodes_of):
        rest = {k: v for k, v in nodes_of.items() if k != r}
        new = level_for_regions(region_nodes, hosp_nodes, rest)
        la = [e for e in base if base[e][0] < 9 and new[e][0] == 9]
        ld = [e for e in base if base[e][0] == 0 and new[e][0] > 0]
        rows.append({'rkey': r, 'n_lost_any': len(la), 'pop65_lost_any': float(sum(pop65.get(e, 0) for e in la)),
                     'n_lost_direct': len(ld), 'pop65_lost_direct': float(sum(pop65.get(e, 0) for e in ld)),
                     'lost_any_eup': ','.join(sorted(la))})
    return base, pd.DataFrame(rows)

In [ ]:
node_xy, nodes_all = build_network(df)
region_nodes, hosp_nodes = assign_nodes(node_xy, sub, hg)
conn, impact = [], []
for sg in T:
    e = sub[sub['시군구명칭'] == sg]
    rn = {a: region_nodes.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
    hn = hosp_nodes.get(sg, set())
    nodes_of = {k: v for k, v in nodes_all.items() if k.startswith(sg + '|')}
    pop65 = e.set_index('adm_cd8')['pop65'].fillna(0).to_dict()
    base, imp = simulate_line_removal(rn, hn, nodes_of, pop65)
    imp.insert(0, 'sigun', sg)
    imp['line'] = imp['rkey'].str.split('|', n=1).str[1]
    impact.append(imp)
    for _, r in e.iterrows():
        lv, direct = base[r['adm_cd8']]
        conn.append({'sigun': sg, 'adm_cd8': r['adm_cd8'], '읍면동': r['읍면동명칭'], 'pop65': r['pop65'],
                     'ratio65': r['ratio65'], 'd_near_m': r['d_near_m'], 'hosp_km': r['hosp_km'],
                     'n_nodes_in': len(rn[r['adm_cd8']]), 'level': lv, 'label': LABEL[lv],
                     'n_direct_lines': len(direct)})
conn = pd.DataFrame(conn)
conn['flag_A_far_stop'] = (conn['n_nodes_in'] == 0) | (conn['d_near_m'] > 1000)
conn['flag_B_no_link'] = (conn['n_nodes_in'] > 0) & (conn['level'] == 9)
conn['flag_transfer'] = conn['level'] == 1
conn['flag_single'] = (conn['level'] == 0) & (conn['n_direct_lines'] == 1)
conn['flag_D_far_hosp'] = conn['hosp_km'] > 15
imp_all = pd.concat(impact, ignore_index=True)

os.makedirs('/content/outputs', exist_ok=True)
conn.to_csv('/content/outputs/eup_connectivity.csv', index=False, encoding='utf-8-sig')
imp_all.to_csv('/content/outputs/line_removal_impact.csv', index=False, encoding='utf-8-sig')
sub.to_file('/content/outputs/stage1.gpkg', driver='GPKG')

print('\n=== 시군별 병원 연결 수준(읍면동 수) ===')
print(conn.groupby(['sigun', 'label']).size().unstack(fill_value=0))
print('\n=== 유형 후보 플래그(읍면동 수) ===')
print

- 출력 깨짐 이슈로 하단 재실행

In [ ]:
pd.set_option('display.width', 200)
fl = [c for c in conn.columns if c.startswith('flag_')]
print('=== 유형 후보 플래그(읍면동 수) ===')
print(conn.groupby('sigun')[fl].sum().to_string())

print('\n=== 노선 제거 시 병원 연결을 잃는 65세 이상 인구 상위 15 ===')
cols = ['sigun', 'line', 'n_lost_any', 'pop65_lost_any', 'n_lost_direct', 'pop65_lost_direct']
print(imp_all.sort_values('pop65_lost_any', ascending=False)[cols].head(15).to_string())

print('\n=== 연결을 잃는 노선이 하나라도 있는 비율 ===')
print((imp_all['n_lost_any'] > 0).groupby(imp_all['sigun']).agg(['sum', 'count']).to_string())

print('\n=== 직통 아님 또는 직통 노선 1개뿐인 읍면동 ===')
x = conn[(conn['label'] != '직통') | (conn['n_direct_lines'] == 1)]
print(x[['sigun', '읍면동', 'pop65', 'hosp_km', 'n_nodes_in', 'label', 'n_direct_lines']].round(1).to_string())

In [ ]:
pd.set_option('display.width', 200)
print('=== 직통을 잃는 65세 이상 인구 상위 15 (직통 기준) ===')
cols = ['sigun', 'line', 'n_lost_direct', 'pop65_lost_direct', 'n_lost_any']
print(imp_all.sort_values(['pop65_lost_direct', 'n_lost_direct'], ascending=False)[cols].head(15).to_string())
print('\n직통을 잃는 노선이 있는 비율(노선 수):')
print((imp_all['n_lost_direct'] > 0).groupby(imp_all['sigun']).agg(['sum', 'count']).to_string())

print('\n=== 김천 아포읍 점검 ===')
ap = sub[(sub['시군구명칭'] == '김천시') & (sub['읍면동명칭'] == '아포읍')]
print(ap[['읍면동명칭', 'pop65', 'n_stops' if 'n_stops' in ap.columns else 'd_near_m', 'd_near_m']].to_string())
sa = gpd.sjoin(stg, ap[['geometry']], predicate='within')
print('정류장 위치 파일에서 아포읍 안 정류장:', len(sa))
us = gpd.GeoDataFrame(df.drop_duplicates(['sigun', 'stop_id']),
                      geometry=gpd.points_from_xy(df.drop_duplicates(['sigun', 'stop_id'])['lon'],
                                                   df.drop_duplicates(['sigun', 'stop_id'])['lat']),
                      crs='EPSG:4326').to_crs('EPSG:5179')
print('노선 파일에서 아포읍 안 정류장:', len(gpd.sjoin(us, ap[['geometry']], predicate='within')))

In [ ]:
pd.set_option('display.width', 200)

In [ ]:
# 직통을 잃는 읍면 이름 확인 (김천 85, 공주 321)
name = sub.set_index('adm_cd8')['읍면동명칭'].to_dict()
for sg, ln in [('김천시', '85'), ('공주시', '321')]:
    e = sub[sub['시군구명칭'] == sg]
    rn = {a: region_nodes.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
    hn = hosp_nodes[sg]
    no = {k: v for k, v in nodes_all.items() if k.startswith(sg + '|')}
    b = level_for_regions(rn, hn, no)
    a = level_for_regions(rn, hn, {k: v for k, v in no.items() if k != f'{sg}|{ln}'})
    print(sg, ln, '번 제거 시 직통 상실:',
          [(name[x], int(sub.set_index('adm_cd8').loc[x, 'pop65']), LABEL[a[x][0]]) for x in b if b[x][0] == 0 and a[x][0] > 0])

In [ ]:
# 읍면별 정류장 위치 파일 수 vs 노선 파일 지점 수
j = gpd.sjoin(stg, sub[['adm_cd8', 'geometry']], predicate='within')
n_loc = j.groupby('adm_cd8').size().rename('stops_loc')
chk = sub[['시군구명칭', '읍면동명칭', 'adm_cd8', 'pop65']].merge(n_loc, on='adm_cd8', how='left').fillna({'stops_loc': 0})
chk['nodes_route'] = chk['adm_cd8'].map(lambda a: len(region_nodes.get(chk.loc[chk['adm_cd8'] == a, '시군구명칭'].iloc[0], {}).get(a, set())))
chk['ratio'] = (chk['nodes_route'] / chk['stops_loc']).round(2)
print('\n=== 위치 파일 대비 노선 파일 지점이 적은 읍면동 (ratio < 0.5) ===')
print(chk[(chk['stops_loc'] > 0) & (chk['ratio'] < 0.5)].sort_values('ratio')[['시군구명칭', '읍면동명칭', 'pop65', 'stops_loc', 'nodes_route', 'ratio']].to_string())
print('\n시군별 평균 비율:'); print(chk.groupby('시군구명칭')['ratio'].describe()[['mean', 'min']].round(2))

# 민감도 확인

In [ ]:
from collections import Counter
name = sub.set_index('adm_cd8')['읍면동명칭'].to_dict()
res = []
for m in (30, 60, 100):
    nxy, nall = build_network(df, m)
    for hr in (300, 500):
        rn_all, hn_all = assign_nodes(nxy, sub, hg, hr)
        lab, single, loss_lines, any_lost = Counter(), [], [], 0
        for sg in T:
            e = sub[sub['시군구명칭'] == sg]
            rn = {a: rn_all.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
            hn = hn_all.get(sg, set())
            no = {k: v for k, v in nall.items() if k.startswith(sg + '|')}
            p65 = e.set_index('adm_cd8')['pop65'].fillna(0).to_dict()
            base, imp = simulate_line_removal(rn, hn, no, p65)
            for a, (lv, d) in base.items():
                lab[(sg, LABEL[lv])] += 1
                if lv == 0 and len(d) == 1:
                    single.append(f"{sg[:2]} {name[a]}")
            any_lost += int((imp['n_lost_any'] > 0).sum())
            for _, r in imp[imp['n_lost_direct'] > 0].iterrows():
                loss_lines.append(f"{sg[:2]} {r['rkey'].split('|', 1)[1]}번({int(r['n_lost_direct'])}곳,{int(r['pop65_lost_direct'])}명)")
        res.append({'병합m': m, '병원m': hr,
                    '직통': sum(v for (s, l), v in lab.items() if l == '직통'),
                    '환승1회': sum(v for (s, l), v in lab.items() if l == '환승1회'),
                    '불가': sum(v for (s, l), v in lab.items() if l == '불가'),
                    '연결상실 노선수': any_lost,
                    '직통상실 노선': ', '.join(loss_lines),
                    '단일노선 읍면': ', '.join(sorted(single))})
pd.set_option('display.max_colwidth', 200, 'display.width', 250)
print(pd.DataFrame(res).to_string(index=False))

# 기존 지표와 순위 비교

In [ ]:
from shapely.ops import unary_union


pd.set_option('display.width', 220)

In [ ]:
# 커버율(500m)을 sub에 추가
cov = unary_union(stg.geometry.buffer(500))
sub['cover500'] = sub.geometry.intersection(cov).area / sub.geometry.area

c = conn.merge(sub[['adm_cd8', 'cover500']], on='adm_cd8')
c['r_near'] = c['d_near_m'].rank(ascending=False, method='min')      # 정류장 멂
c['r_cov']  = c['cover500'].rank(ascending=True,  method='min')      # 커버율 낮음
c['r_hosp'] = c['hosp_km'].rank(ascending=False, method='min')       # 병원 멂
c['r_old']  = c['ratio65'].rank(ascending=False, method='min')       # 고령 비율 높음

V = c[c['flag_single'] | c['flag_transfer']]
## 노선 기반 취약 후보
print('=== 노선 기반 취약 후보와 기존 지표 순위 (52개 중, 1=가장 취약) ===')
print(V[['sigun', '읍면동', 'pop65', 'label', 'n_direct_lines', 'r_near', 'r_cov', 'r_hosp', 'r_old']].round(0).to_string(index=False))

print('\n=== 상위 k 겹침: 후보 %d곳 중 기존 지표 상위 k에 든 수 (우연 기대) ===' % len(V))
for k in (5, 10, 15, 20):
    out = {}
    for col, nm in [('r_near', '정류장거리'), ('r_cov', '커버율'), ('r_hosp', '병원거리'), ('r_old', '고령비율')]:
        out[nm] = f"{int((V[col] <= k).sum())} ({len(V) * k / len(c):.1f})"
    any3 = int(((V['r_near'] <= k) | (V['r_cov'] <= k) | (V['r_hosp'] <= k)).sum())
    print(f"k={k}:", out, f"| 거리계 3개 중 하나라도 상위k: {any3}/{len(V)}")

# 이미지2

In [ ]:
import subprocess, glob
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

In [ ]:
subprocess.run('apt-get -qq -y install fonts-nanum', shell=True, capture_output=True)
for f in glob.glob('/usr/share/fonts/truetype/nanum/*.ttf'):
    fm.fontManager.addfont(f)
mpl.rcParams.update({'font.family': 'NanumGothic', 'axes.unicode_minus': False, 'savefig.dpi': 200,
                     'axes.spines.top': False, 'axes.spines.right': False})
INK, INK2 = '#0b0b0b', '#52514e'
C_SINGLE, C_TRANS, C_OTHER, C_NA = '#eb6834', '#2a78d6', '#e6e5e1', '#c9c8c3'
COL = {'단일 노선 의존': C_SINGLE, '환승 필요': C_TRANS, '노선 데이터 미확인': C_NA, '그 외': C_OTHER}

def cat_of(r):
    if r['flag_single']: return '단일 노선 의존'
    if r['flag_transfer']: return '환승 필요'
    if r['n_nodes_in'] == 0: return '노선 데이터 미확인'
    return '그 외'
conn['cat'] = conn.apply(cat_of, axis=1)
S = sub.merge(conn[['adm_cd8', 'cat']], on='adm_cd8')

In [ ]:
## 취약 읍면 지도
fig, axs = plt.subplots(1, 3, figsize=(13, 5.2))
for ax, sg in zip(axs, T):
    d = S[S['시군구명칭'] == sg]
    d.plot(ax=ax, color=d['cat'].map(COL).tolist(), edgecolor='white', linewidth=0.8)
    na = d[d['cat'] == '노선 데이터 미확인']
    if len(na):
        na.plot(ax=ax, color=C_NA, edgecolor=INK2, linewidth=0.8, hatch='///')
    hg[hg['시군구'] == sg].plot(ax=ax, color=INK, marker='P', markersize=50, zorder=5)
    for _, r in d[d['cat'] != '그 외'].iterrows():
        p = r.geometry.representative_point()
        ax.annotate(r['읍면동명칭'], (p.x, p.y), ha='center', va='center', fontsize=8.5, color=INK, zorder=6,
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))
    ax.set_title(sg, fontsize=12, color=INK); ax.set_axis_off()
handles = [Patch(fc=C_SINGLE, label='병원 직통 노선 1개뿐 (단일 노선 의존)'),
           Patch(fc=C_TRANS, label='병원까지 환승 필요'),
           Patch(fc=C_NA, ec=INK2, hatch='///', label='노선 데이터 미확인(아포읍)'),
           Patch(fc=C_OTHER, label='그 외 (병원 직통 2개 이상)'),
           Line2D([], [], marker='P', color='w', markerfacecolor=INK, markersize=9, label='병원(병원·종합병원)')]
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, fontsize=9)
fig.text(0.01, 0.005, '병원 반경 300m 기준. 경계 2025-06-30, 정류장 2025-10-31, 병원 2026-10-12, 노선 API 수집본.',
         fontsize=7.5, color=INK2)
fig.tight_layout(rect=(0, 0.09, 1, 1))
fig.savefig('/content/outputs/fig1_map.png', bbox_inches='tight', facecolor='white'); plt.show()

In [ ]:
## 직통-1, 65세 인구
name = sub.set_index('adm_cd8')['읍면동명칭'].to_dict()
pp = sub.set_index('adm_cd8')['pop65'].to_dict()
items = []
for _, r in imp_all[imp_all['n_lost_direct'] > 0].sort_values('pop65_lost_direct').iterrows():
    sg, ln = r['sigun'], r['line']
    e = sub[sub['시군구명칭'] == sg]
    rn = {a: region_nodes.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
    no = {k: v for k, v in nodes_all.items() if k.startswith(sg + '|')}
    b = level_for_regions(rn, hosp_nodes[sg], no)
    a2 = level_for_regions(rn, hosp_nodes[sg], {k: v for k, v in no.items() if k != f'{sg}|{ln}'})
    segs = [(name[x], int(pp[x])) for x in b if b[x][0] == 0 and a2[x][0] > 0]
    items.append((f"{sg[:2]} {ln}번", segs))
fig, ax = plt.subplots(figsize=(10, 2.2 + 1.1 * len(items)))
for i, (lab, segs) in enumerate(items):
    left = 0
    for nm, v in sorted(segs, key=lambda t: -t[1]):
        ax.barh(i, v, left=left, height=0.5, color=C_SINGLE, edgecolor='white', linewidth=2)
        ax.text(left + v / 2, i, f"{nm}\n{v:,}명", ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        left += v
    ax.text(left + 40, i, f"합계 {left:,}명", va='center', fontsize=10, color=INK)
ax.set_yticks(range(len(items))); ax.set_yticklabels([t[0] for t in items], fontsize=11)
ax.set_xlabel('이 노선이 사라지면 병원 직통을 잃는 읍면의 65세 이상 인구(명)', color=INK2)
ax.set_xlim(0, max(sum(v for _, v in s) for _, s in items) * 1.25)
ax.text(0, -0.9, f"나머지 {len(imp_all) - len(items)}개 노선은 하나를 빼도 직통을 잃는 읍면이 없음(0명). 병원 반경 500m로 넓히면 김천 85번 영향은 사라지고 공주 321번만 남음.",
        fontsize=8.5, color=INK2, transform=ax.transData)
fig.savefig('/content/outputs/fig2_route_removal.png', bbox_inches='tight', facecolor='white'); plt.show()

In [ ]:
#정류장vs병원(거리)
fig, ax = plt.subplots(figsize=(8.5, 6))
g0 = conn[conn['cat'] == '그 외']
ax.scatter(g0['d_near_m'], g0['hosp_km'], s=28, color='#b9b8b2', edgecolor='white', linewidth=0.8, zorder=2)
for k in ('단일 노선 의존', '환승 필요'):
    g1 = conn[conn['cat'] == k]
    ax.scatter(g1['d_near_m'], g1['hosp_km'], s=70, color=COL[k], edgecolor='white', linewidth=1.2, zorder=3, label=k)
na = conn[conn['cat'] == '노선 데이터 미확인']
ax.scatter(na['d_near_m'], na['hosp_km'], s=70, facecolor='white', edgecolor=INK2, linewidth=1.5, zorder=3, label='노선 데이터 미확인')
offs = [(7, 6), (7, -13), (-7, 8), (-7, -14)]
for i, (_, r) in enumerate(conn[conn['cat'] != '그 외'].sort_values('hosp_km').iterrows()):
    dx, dy = offs[i % 4]
    ax.annotate(f"{r['sigun'][:2]} {r['읍면동']}", (r['d_near_m'], r['hosp_km']), xytext=(dx, dy),
                textcoords='offset points', fontsize=9, color=INK, ha='left' if dx > 0 else 'right')
ax.axvline(conn['d_near_m'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.axhline(conn['hosp_km'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.set_xlabel('가장 가까운 정류장까지 거리(m, 읍면동 대표점)', color=INK2)
ax.set_ylabel('같은 시군 병원까지 직선거리(km)', color=INK2)
ax.legend(frameon=False, loc='upper right', fontsize=9)
ax.text(0.01, -0.12, '점선은 52개 읍면동 중앙값. 회색 점은 나머지 읍면동.', transform=ax.transAxes, fontsize=8, color=INK2)
fig.savefig('/content/outputs/fig3_stop_vs_hospital.png', bbox_inches='tight', facecolor='white'); plt.show()
print('저장:', glob.glob('/content/outputs/fig*.png'))

In [ ]:
from google.colab import files
for f in ['fig1_map.png', 'fig2_route_removal.png', 'fig3_stop_vs_hospital.png']:
    files.download(f'/content/outputs/{f}')

# 아포읍 점검 코드

In [ ]:
ap = sub[(sub['시군구명칭'] == '김천시') & (sub['읍면동명칭'] == '아포읍')]
sa = gpd.sjoin(stg, ap[['geometry']], predicate='within')
cols = [c for c in ['도시명', '관리도시명', '정류장번호', '정류장명', '위도', '경도'] if c in sa.columns]
print('아포읍 안 정류장', len(sa), '개'); print(sa[cols].to_string(index=False))
if '관리도시명' in stg.columns:
    print('\n김천시 정류장의 관리도시명 분포:')
    print(stg[stg['도시명'].astype(str).str.contains('김천')]['관리도시명'].value_counts().to_string())
names = set(sa['정류장명'].astype(str)) if '정류장명' in sa.columns else set()
hit = df[df['stop_name'].astype(str).isin(names)]
print('\n노선 파일에서 이 이름과 같은 정류장:', hit[['sigun', 'route_no', 'stop_name']].drop_duplicates().head(20).to_string(index=False) if len(hit) else '없음')

In [ ]:
ids = set(sa['정류장번호'].astype(str))
hit = df[df['stop_id'].astype(str).isin(ids)]
print('노선 파일에서 아포읍 정류장 ID와 일치:', len(hit), '행')
print(hit[['sigun', 'route_no', 'stop_id', 'stop_name']].drop_duplicates().to_string(index=False) if len(hit) else '없음')

In [ ]:
import numpy as np
o = sa[sa['정류장명'] == '아포읍사무소앞'].geometry.iloc[0]
cand = us[us['sigun'] == '김천시'].copy()
cand['d'] = cand.geometry.distance(o)
print('\n아포읍사무소앞에서 가장 가까운 노선 파일 정류장 5개:')
print(cand.nsmallest(5, 'd')[['stop_name', 'stop_id', 'd']].round(0).to_string(index=False))

# 그림 수정

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 5.6))
for ax, sg in zip(axs, T):
    d = S[S['시군구명칭'] == sg]
    d.plot(ax=ax, color=d['cat'].map(COL).tolist(), edgecolor='white', linewidth=0.8)
    na = d[d['cat'] == '노선 데이터 미확인']
    if len(na):
        na.plot(ax=ax, color=C_NA, edgecolor=INK2, linewidth=0.8, hatch='///')
    hg[hg['시군구'] == sg].plot(ax=ax, color=INK, marker='P', markersize=50, zorder=5)
    for _, r in d[d['cat'] != '그 외'].iterrows():
        p = r.geometry.representative_point()
        ax.annotate(r['읍면동명칭'], (p.x, p.y), ha='center', va='center', fontsize=8.5, color=INK, zorder=6,
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))
    if (d['cat'] != '그 외').sum() == 0:
        ax.text(0.5, 0.04, '해당 읍면 없음\n(모든 읍면이 병원 직통 2개 이상)', transform=ax.transAxes,
                ha='center', fontsize=9.5, color=INK2)
    ax.set_title(sg, fontsize=12, color=INK, y=1.0); ax.set_axis_off()
handles = [Patch(fc=C_SINGLE, label='병원 직통 노선 1개뿐 (단일 노선 의존)'),
           Patch(fc=C_TRANS, label='병원까지 환승 필요'),
           Patch(fc=C_NA, ec=INK2, hatch='///', label='노선 데이터 미확인(아포읍)'),
           Patch(fc=C_OTHER, label='그 외 (병원 직통 2개 이상)'),
           Line2D([], [], marker='P', color='w', markerfacecolor=INK, markersize=9, label='병원(병원·종합병원)')]
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.06), ncol=3, frameon=False, fontsize=9)
fig.text(0.01, 0.01, '병원 반경 300m 기준. 경계 2025-06-30, 정류장 2025-10-31, 병원 2026-10-12, 인구 2026-09-30, 노선 API 수집본.',
         fontsize=7.5, color=INK2)
fig.tight_layout(rect=(0, 0.14, 1, 1))
fig.savefig('/content/outputs/fig1_map.png', bbox_inches='tight', facecolor='white'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
g0 = conn[conn['cat'] == '그 외']
ax.scatter(g0['d_near_m'], g0['hosp_km'], s=28, color='#b9b8b2', edgecolor='white', linewidth=0.8, zorder=2)
for k in ('단일 노선 의존', '환승 필요'):
    g1 = conn[conn['cat'] == k]
    ax.scatter(g1['d_near_m'], g1['hosp_km'], s=70, color=COL[k], edgecolor='white', linewidth=1.2, zorder=3, label=k)
na = conn[conn['cat'] == '노선 데이터 미확인']
ax.scatter(na['d_near_m'], na['hosp_km'], s=70, facecolor='white', edgecolor=INK2, linewidth=1.5, zorder=3, label='노선 데이터 미확인')
OFF = {'탄천면': (10, 4, 'left'), '반포면': (-9, 7, 'right'), '구성면': (9, -13, 'left'),
       '지례면': (9, 6, 'left'), '부항면': (-9, 7, 'right'), '대덕면': (-9, 8, 'right'),
       '증산면': (-9, -15, 'right'), '아포읍': (-9, 9, 'right')}
for _, r in conn[conn['cat'] != '그 외'].iterrows():
    dx, dy, ha = OFF.get(r['읍면동'], (8, 6, 'left'))
    ax.annotate(f"{r['sigun'][:2]} {r['읍면동']}", (r['d_near_m'], r['hosp_km']), xytext=(dx, dy),
                textcoords='offset points', fontsize=9, color=INK, ha=ha)
ax.axvline(conn['d_near_m'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.axhline(conn['hosp_km'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.set_ylim(-1, 31)
ax.set_xlabel('가장 가까운 정류장까지 거리(m, 읍면동 대표점)', color=INK2)
ax.set_ylabel('같은 시군 병원까지 직선거리(km)', color=INK2)
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.text(0.01, -0.12, '점선은 52개 읍면동 중앙값. 회색 점은 나머지 읍면동.', transform=ax.transAxes, fontsize=8, color=INK2)
fig.savefig('/content/outputs/fig3_stop_vs_hospital.png', bbox_inches='tight', facecolor='white'); plt.show()

In [ ]:
from google.colab import files
for f in ['fig1_map.png', 'fig2_route_removal.png', 'fig3_stop_vs_hospital.png']:
    files.download(f'/content/outputs/{f}')